In [0]:
# Load the latest curated Delta tables as Spark DataFrames for referential integrity validation
customers_df = spark.read.format("delta").load(
    "/Volumes/workspace/retail_schema/clean/Customers"
)

orders_df = spark.read.format("delta").load(
    "/Volumes/workspace/retail_schema/clean/Orders"
)

payments_df = spark.read.format("delta").load(
    "/Volumes/workspace/retail_schema/clean/Payments"
)

products_df = spark.read.format("delta").load(
    "/Volumes/workspace/retail_schema/clean/Products"
)

orders_items_df = spark.read.format("delta").load(
    "/Volumes/workspace/retail_schema/clean/OrderItems"
)


In [0]:
# Define referential integrity rules for all key table relationships
# Each rule links child and parent tables via one or more join columns and sets a violation threshold
ri_config = [
    {
        "child_table": "orders",
        "parent_table": "customers",
        "child_df": orders_df,
        "parent_df": customers_df,
        "join_columns": ["customer_id"],
        "threshold": 0
    },
    {
        "child_table": "order_items",
        "parent_table": "orders",
        "child_df": orders_items_df,
        "parent_df": orders_df,
        "join_columns": ["order_id"],
        "threshold": 0
    },
    {
        "child_table": "payments",
        "parent_table": "orders",
        "child_df": payments_df,
        "parent_df": orders_df,
        "join_columns": ["order_id"],
        "threshold": 0
    },
    {
        "child_table": "order_items",
        "parent_table": "products",
        "child_df": orders_items_df,
        "parent_df": products_df,
        "join_columns": ["product_id"],
        "threshold": 0
    }
]


In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, current_timestamp

def run_referential_integrity_checks(ri_config):
    """
    Loops over each RI rule. For each:
    - Filters nulls on join columns
    - Left anti joins child to parent (finds orphans)
    - Counts violations and adds metrics row (Pass/Fail by threshold)
    Returns metrics DataFrame with timestamp.
    """
    results = []
    for rule in ri_config:
        child_df = rule["child_df"]
        parent_df = rule["parent_df"]
        join_columns = rule["join_columns"]
        threshold = rule["threshold"]
        child_table = rule["child_table"]
        parent_table = rule["parent_table"]

        # Remove nulls in join columns (recommended for RI check accuracy)
        filtered_child = child_df
        for c in join_columns:
            filtered_child = filtered_child.filter(col(c).isNotNull())

        # Left anti join to find invalid references (orphans)
        invalid_df = (
            filtered_child
            .select(*join_columns)
            .join(
                parent_df.select(*join_columns),
                on=join_columns,
                how="left_anti"
            )
        )

        fail_count = invalid_df.count()
        status = "Pass" if fail_count <= threshold else "Fail"

        results.append(Row(
            table_name=child_table,
            column_name="NA",
            check_name=f"{child_table}_to_{parent_table}_referential_integrity_check",
            metric_type="count",
            metric_value=fail_count,
            threshold=threshold,
            status=status
        ))

    # Aggregate all RI results, add run timestamp
    metrics_df = spark.createDataFrame(results) \
        .withColumn("run_time", current_timestamp())
    return metrics_df


In [0]:
# Execute all referential integrity checks and generate summary metrics DataFrame
metrics_df = run_referential_integrity_checks(ri_config)
metrics_df


DataFrame[table_name: string, column_name: string, check_name: string, metric_type: string, metric_value: bigint, threshold: bigint, status: string, run_time: timestamp]